# Лабораторна робота №5 — Розпізнавання облич

**Тиждень 9:** Класичні алгоритми ML — SVM, Random Forest, KNN  
**Тиждень 10:** Глибоке навчання — згорткова нейронна мережа (CNN)

---

## Мета

1. Реалізувати та порівняти три класичних класифікатори (SVM, Random Forest, KNN) для задачі розпізнавання облич.
2. Побудувати та навчити CNN для тієї ж задачі, порівняти результати з класичними підходами.

## Теоретична довідка

### Класичний підхід (Week 9)

Конвеєр обробки ознак:
```
зображення → detect face → crop → resize(64×64) → flatten → StandardScaler → PCA → classifier
```

- **SVM (Support Vector Machine)** з RBF-ядром — максимізує відступ між класами у просторі ознак.
- **Random Forest** — ансамбль дерев рішень з голосуванням більшості.
- **KNN (k-Nearest Neighbors)** — класифікація за k найближчими сусідами у просторі PCA.

### Глибоке навчання (Week 10)

**CNN** (Convolutional Neural Network) — ієрархічно вивчає ознаки безпосередньо з пікселів:
```
Input → [Conv→BN→ReLU→Pool] × 3 → GlobalAvgPool → Dropout → Dense(softmax)
```

## 0. Налаштування середовища

In [ ]:
import sys
from pathlib import Path

# Додаємо src/ до шляху пошуку модулів
PROJECT_ROOT = Path(".").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("NumPy:", np.__version__)

## 1. Підготовка датасету

Для демонстрації використаємо **синтетичний датасет** (4 особи, 20 зображень кожна).  
Щоб використати реальний датасет — вкажіть шлях у змінній `DATASET_DIR`.

> **Рекомендовані датасети:**
> - [AT&T (ORL) Faces](https://cam-orl.co.uk/facedatabase.html) — 40 осіб × 10 зображень
> - [LFW (Labeled Faces in the Wild)](http://vis-www.cs.umass.edu/lfw/) — 5749 осіб

In [ ]:
# Вкажіть шлях до реального датасету або залиште None для синтетичних даних
DATASET_DIR = None  # наприклад: "data/lfw/" або "data/att_faces/"

if DATASET_DIR is None:
    print("DATASET_DIR не задано — буде використано синтетичний датасет.")
else:
    p = Path(DATASET_DIR)
    classes = [d.name for d in p.iterdir() if d.is_dir()]
    print(f"Датасет: {DATASET_DIR}")
    print(f"Кількість класів (осіб): {len(classes)}")
    print(f"Перші 10: {classes[:10]}")

---

# Тиждень 9: Класичні алгоритми ML (SVM, Random Forest, KNN)

## 2. Навчання класичних класифікаторів

In [ ]:
from cv_human_search.classical_recognition import ClassicalFaceRecognizer

# Якщо DATASET_DIR=None — генеруємо синтетичний датасет
if DATASET_DIR is None:
    import tempfile
    from cv_human_search.pipeline import _make_synthetic_dataset
    tmp_dir = tempfile.mkdtemp(prefix="cv_lab5_")
    _make_synthetic_dataset(tmp_dir, n_classes=4, images_per_class=20)
    train_dir = tmp_dir
    print(f"Синтетичний датасет створено: {tmp_dir}")
else:
    train_dir = DATASET_DIR

# Ініціалізуємо класифікатор
recognizer = ClassicalFaceRecognizer(
    target_size=(64, 64),   # розмір зображення після нормалізації
    n_neighbors=5,          # для KNN
    n_estimators=150,       # дерева для Random Forest
    random_state=42,
)

# Навчаємо всі три класифікатори
report = recognizer.train_from_directory(train_dir, test_size=0.25)
print("\nНавчання завершено!")

## 3. Результати: порівняльний звіт

In [ ]:
print(report.summary())

## 4. Візуалізація: точність і матриця помилок

На графіку зображено:
- **Верхній рядок:** порівняння точності (accuracy) трьох класифікаторів
- **Нижній рядок:** матриця помилок (confusion matrix) для кожного класифікатора — по рядку нормована, щоб кольори були порівнянними між класами різного розміру

In [ ]:
report.plot_comparison()

## 5. Аналіз результатів (Week 9)

Зведена таблиця точності:

In [ ]:
import pandas as pd

rows = [
    {
        "Класифікатор": m.name,
        "Точність (accuracy)": f"{m.accuracy:.4f}",
        "Accuracy %": f"{m.accuracy * 100:.2f}%",
    }
    for m in report.metrics
]
df = pd.DataFrame(rows).set_index("Класифікатор")
display(df)

**Висновок (Week 9):**

- **SVM (RBF)** — зазвичай показує найкращі результати завдяки ефективній побудові гіперплощини у просторі ознак PCA.
- **Random Forest** — стабільний ансамблевий метод, менш чутливий до налаштувань гіперпараметрів.
- **KNN** — простий, але ефективний базовий метод; якість залежить від кількості сусідів `k` та розмірності простору ознак.

Зменшення розмірності через **PCA** перед класифікацією суттєво покращує швидкість навчання та стабільність результатів.

---

# Тиждень 10: Глибоке навчання — CNN

## 6. Архітектура та навчання CNN

In [ ]:
from cv_human_search.cnn_recognition import CNNFaceRecognizer

cnn = CNNFaceRecognizer(input_size=(64, 64))

history = cnn.train_from_directory(
    train_dir,
    epochs=20,
    batch_size=16,
    patience=6,
    test_size=0.25,
)

## 7. Крива навчання (Loss та Accuracy)

Графіки дозволяють виявити:
- **Overfitting:** тренувальна точність зростає, але валідаційна — ні
- **Underfitting:** обидві криві не сходяться до прийнятного рівня
- **Early stopping:** момент, коли навчання зупиняється автоматично

In [ ]:
cnn.plot_history()

## 8. Статистика навчання CNN

In [ ]:
print(history.summary())

print(f"\nКількість епох до зупинки: {len(history.train_loss)}")
print(f"Найкраща val_accuracy: {max(history.val_acc):.4f}")
print(f"Найкраща val_loss:     {min(history.val_loss):.4f}")

## 9. Візуалізація карт ознак (Feature Maps)

Карти ознак показують, **що саме виявляє кожен фільтр** згорткового шару.  
Шар `conv1` — низькорівневі ознаки (краї, текстури).  
Шар `conv2`, `conv3` — більш абстрактні ознаки (частини обличчя).

In [ ]:
import cv2
import numpy as np

# Створюємо тестове зображення (сірий еліпс — спрощена «особа»)
test_img = np.zeros((96, 96), dtype=np.uint8)
cv2.ellipse(test_img, (48, 48), (35, 42), 0, 0, 360, color=180, thickness=-1)
test_img_bgr = cv2.cvtColor(test_img, cv2.COLOR_GRAY2BGR)

# Шар conv1 — перший згортковий шар
cnn.visualize_feature_maps(test_img_bgr, layer_name="conv1")

## 10. Порівняння всіх підходів: Classical ML vs CNN

In [ ]:
all_results = [
    (m.name, m.accuracy) for m in report.metrics
] + [("CNN", max(history.val_acc))]

names = [r[0] for r in all_results]
accs  = [r[1] for r in all_results]
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, accs, color=colors, width=0.5,
              edgecolor="white", linewidth=0.8)

for bar, acc in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        acc + 0.015,
        f"{acc:.3f}",
        ha="center", va="bottom", fontsize=12, fontweight="bold",
    )

ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Порівняння всіх методів: Classical ML vs CNN", fontsize=14)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax.grid(axis="y", alpha=0.35)
ax.spines[["top", "right"]].set_visible(False)

# Додаємо межу між класичними і CNN методами
ax.axvline(x=2.5, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax.text(1.0, 1.08, "Класичні ML", ha="center", color="gray", fontsize=10)
ax.text(3.0, 1.08, "Deep Learning", ha="center", color="gray", fontsize=10)

plt.tight_layout()
plt.show()

# Таблиця
print("\n{:<20} {:>12}".format("Метод", "Accuracy"))
print("-" * 34)
for name, acc in all_results:
    print("{:<20} {:>12.4f}".format(name, acc))

## 11. Підсумки та висновки

### Тиждень 9 — Класичне машинне навчання

| Метод | Переваги | Недоліки |
|---|---|---|
| SVM (RBF) | Висока точність на малих датасетах, ефективна апроксимація меж класів | Повільне навчання при великих датасетах, потребує підбору C і γ |
| Random Forest | Стійкий до шуму, не потребує нормалізації ознак | Менш точний ніж SVM при малій кількості даних |
| KNN | Простота, відсутність етапу навчання | Повільна класифікація при великому датасеті, чутливий до шуму |

**Попередня обробка:** Зменшення розмірності через PCA дозволяє:
- Прискорити навчання у 5–10 разів
- Зменшити перенавчання
- Усунути кореляції між пікселями

### Тиждень 10 — Глибоке навчання (CNN)

- CNN **не потребує ручного видобутку ознак** — вивчає їх самостійно з пікселів
- Завдяки **аугментації даних** (flip, rotate, brightness) модель стійкіша до варіацій освітлення та пози
- **Early stopping** запобігає перенавчанню
- Карти ознак демонструють, що мережа дійсно вивчила значущі локальні патерни (краї, регіони обличчя)

### Загальний висновок

На малих датасетах класичні методи (особливо SVM) можуть конкурувати з CNN.  
На великих датасетах CNN значно перевершує класичні підходи завдяки можливості вивчати ієрархічні ознаки.

In [ ]:
# Очищаємо тимчасовий датасет
if DATASET_DIR is None:
    import shutil
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"Тимчасовий датасет видалено: {tmp_dir}")